<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Setup and search grid

import mpmath as mp
import csv
from collections import defaultdict

# Working precision for the initial search
mp.mp.dps = 40

SIGMA_STARTS = [
    mp.mpf("0.15"), mp.mpf("0.30"),
    mp.mpf("0.45"), mp.mpf("0.55"),
    mp.mpf("0.70"), mp.mpf("0.85")
]

T_STARTS = [
    mp.mpf(str(t)) for t in
    [12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60]
]

# Search settings
MAX_STEPS = 80
SEARCH_TOL = mp.mpf("1e-30")
DEDUP_TOL = mp.mpf("1e-20")

print("Precision:", mp.mp.dps, "decimal digits")
print("Starting points:", len(SIGMA_STARTS) * len(T_STARTS))


Precision: 40 decimal digits
Starting points: 78


In [3]:
#2-D search

def complex_distance(a, b):
    return abs(a - b)


def add_if_new(roots, candidate, tolerance=DEDUP_TOL):
    for old in roots:
        if complex_distance(old, candidate) < tolerance:
            return False
    roots.append(candidate)
    return True


found_roots = []
attempt_log = []

for sigma in SIGMA_STARTS:
    for t in T_STARTS:
        z0 = mp.mpc(sigma, t)
        z1 = z0 + mp.mpc("0.001", "0.001")

        record = {
            "sigma_start": mp.nstr(sigma, 12),
            "t_start": mp.nstr(t, 12),
            "status": "failed",
            "root": "",
            "residual": "",
            "error": ""
        }

        try:
            root = mp.findroot(
                mp.zeta,
                (z0, z1),
                solver="secant",
                tol=SEARCH_TOL,
                maxsteps=MAX_STEPS,
                verify=True
            )

            residual = abs(mp.zeta(root))

            record["status"] = "converged"
            record["root"] = mp.nstr(root, 35)
            record["residual"] = mp.nstr(residual, 8)

            # Keep only roots in the search region.
            if (
                mp.mpf("0.1") <= mp.re(root) <= mp.mpf("0.9")
                and mp.mpf("10") <= mp.im(root) <= mp.mpf("60")
                and residual < mp.mpf("1e-25")
            ):
                add_if_new(found_roots, root)

        except Exception as exc:
            record["error"] = str(exc)[:180]

        attempt_log.append(record)

print("Total attempts:", len(attempt_log))
print("Successful convergences:", sum(
    r["status"] == "converged" for r in attempt_log
))
print("Distinct roots retained:", len(found_roots))

for i, root in enumerate(found_roots, 1):
    print(
        i,
        "sigma =", mp.nstr(mp.re(root), 25),
        "t =", mp.nstr(mp.im(root), 25),
        "|zeta| =", mp.nstr(abs(mp.zeta(root)), 8)
    )


Total attempts: 78
Successful convergences: 50
Distinct roots retained: 10
1 sigma = 0.5 t = 14.13472514173469379045725 |zeta| = 5.0942619e-47
2 sigma = 0.5 t = 21.02203963877155499262848 |zeta| = 2.1986867e-46
3 sigma = 0.5 t = 25.01085758014568876321379 |zeta| = 2.9923501e-46
4 sigma = 0.5 t = 32.93506158773918969066237 |zeta| = 1.7287236e-46
5 sigma = 0.5 t = 43.32707328091499951949612 |zeta| = 1.8996014e-46
6 sigma = 0.5 t = 40.91871901214749518739813 |zeta| = 6.3707359e-47
7 sigma = 0.5 t = 48.00515088116715972794247 |zeta| = 1.552675e-46
8 sigma = 0.5 t = 56.44624769706339480436776 |zeta| = 3.0826201e-46
9 sigma = 0.5 t = 59.34704400260235307965365 |zeta| = 6.5178804e-46
10 sigma = 0.5 t = 30.4248761258595132103119 |zeta| = 1.5687407e-46


In [4]:
#Recalculate candidates at 80 digits

mp.mp.dps = 80

validated_roots = []

for original_root in found_roots:
    try:
        # Refine the 40-digit candidate at 80-digit precision.
        guess = mp.mpc(original_root)
        refined = mp.findroot(
            mp.zeta,
            (guess, guess + mp.mpc("1e-8", "1e-8")),
            solver="secant",
            tol=mp.mpf("1e-65"),
            maxsteps=100,
            verify=True
        )

        residual = abs(mp.zeta(refined))

        in_region = (
            mp.mpf("0.1") <= mp.re(refined) <= mp.mpf("0.9")
            and mp.mpf("10") <= mp.im(refined) <= mp.mpf("60")
        )

        passed = in_region and residual < mp.mpf("1e-50")

        validated_roots.append({
            "root": refined,
            "residual": residual,
            "passed": passed
        })

    except Exception as exc:
        validated_roots.append({
            "root": original_root,
            "residual": mp.inf,
            "passed": False
        })
        print("Refinement failed:", str(exc)[:180])

print("\n80-digit refinement results")

for i, item in enumerate(validated_roots, 1):
    root = item["root"]
    print("\nCandidate", i)
    print("sigma:", mp.nstr(mp.re(root), 60))
    print("t:    ", mp.nstr(mp.im(root), 60))
    print("|zeta|:", mp.nstr(item["residual"], 12))
    print("Passed numerical checks:", item["passed"])



80-digit refinement results

Candidate 1
sigma: 0.5
t:     14.1347251417346937904572519835624702707842571156992431756856
|zeta|: 4.23557065274e-88
Passed numerical checks: True

Candidate 2
sigma: 0.5
t:     21.0220396387715549926284795938969027773343405249027817546295
|zeta|: 9.05670121407e-87
Passed numerical checks: True

Candidate 3
sigma: 0.5
t:     25.0108575801456887632137909925628218186595496725579966724965
|zeta|: 1.47769664681e-86
Passed numerical checks: True

Candidate 4
sigma: 0.5
t:     32.93506158773918969066236896407490348881271560351703900928
|zeta|: 2.19916053928e-86
Passed numerical checks: True

Candidate 5
sigma: 0.5
t:     43.3270732809149995194961221654068057826456683718368714468789
|zeta|: 2.30444632715e-86
Passed numerical checks: True

Candidate 6
sigma: 0.5
t:     40.9187190121474951873981269146332543957261659627772795361613
|zeta|: 3.59085840474e-86
Passed numerical checks: True

Candidate 7
sigma: 0.5
t:     48.005150881167159727942472749427516041686844001

In [5]:
#Experiment -2 :Check against known zeros

mp.mp.dps = 80

# Reference zeros, generated independently of our root search.
reference_zeros = [
    mp.zetazero(n) for n in range(1, 101)
]

print("Reference zeros generated:", len(reference_zeros))

for i, item in enumerate(validated_roots, 1):
    rho = item["root"]

    nearest = min(
        reference_zeros,
        key=lambda z: abs(rho - z)
    )
    nearest_distance = abs(rho - nearest)

    # Functional-equation reflection and conjugation checks.
    reflected = 1 - rho
    conjugate = mp.conj(rho)

    reflected_residual = abs(mp.zeta(reflected))
    conjugate_residual = abs(mp.zeta(conjugate))

    print("\nCandidate", i)
    print("Distance to nearest reference zero:",
          mp.nstr(nearest_distance, 12))
    print("Nearest reference zero:",
          mp.nstr(nearest, 35))
    print("Reflected residual:",
          mp.nstr(reflected_residual, 12))
    print("Conjugate residual:",
          mp.nstr(conjugate_residual, 12))


Reference zeros generated: 100

Candidate 1
Distance to nearest reference zero: 6.66251596028e-81
Nearest reference zero: (0.5 + 14.13472514173469379045725198356247j)
Reflected residual: 5.28444794308e-81
Conjugate residual: 5.28444794308e-81

Candidate 2
Distance to nearest reference zero: 1.46442222418e-80
Nearest reference zero: (0.5 + 21.022039638771554992628479593896903j)
Reflected residual: 1.66481178064e-80
Conjugate residual: 1.66481178064e-80

Candidate 3
Distance to nearest reference zero: 5.69500033911e-81
Nearest reference zero: (0.5 + 25.010857580145688763213790992562822j)
Reflected residual: 7.81197140198e-81
Conjugate residual: 7.81197140198e-81

Candidate 4
Distance to nearest reference zero: 2.32024228011e-80
Nearest reference zero: (0.5 + 32.935061587739189690662368964074903j)
Reflected residual: 3.2068507927e-80
Conjugate residual: 3.2068507927e-80

Candidate 5
Distance to nearest reference zero: 9.37087121375e-81
Nearest reference zero: (0.5 + 43.3270732809149995194

In [7]:
#Reproducible research log

# Log every starting point, including failed attempts.
with open("session02_search_attempts.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=attempt_log[0].keys())
    writer.writeheader()
    writer.writerows(attempt_log)

# Export refined candidates and their checks.
with open("session02_validated_candidates.csv", "w", newline="") as f:
    fields = [
        "candidate",
        "sigma",
        "t",
        "residual",
        "numerically_validated"
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    for i, item in enumerate(validated_roots, 1):
        root = item["root"]
        writer.writerow({
            "candidate": i,
            "sigma": mp.nstr(mp.re(root), 70),
            "t": mp.nstr(mp.im(root), 70),
            "residual": mp.nstr(item["residual"], 15),
            "numerically_validated": item["passed"]
        })

print("Saved:")
print(" - session02_search_attempts.csv")
print(" - session02_validated_candidates.csv")


Saved:
 - session02_search_attempts.csv
 - session02_validated_candidates.csv
